In [0]:
# Databricks notebook source
# File: bronze_to_silver/01_bronze_to_silver.py
# Project: MIGRATION-DBX-001
# Purpose: Read Bronze Parquet → enforce schema → DQ → MERGE into Silver Delta
# Triggered by: Databricks Job (job_bronze_to_silver)
# =============================================================================

In [0]:
# COMMAND ----------
# MAGIC %md
# MAGIC ## Bronze → Silver
# MAGIC **Flow:** Read Bronze Parquet → Schema enforcement → Dedup → DQ rules → Delta MERGE / Quarantine
# MAGIC
# MAGIC **Run parameters** (passed by Databricks Job):
# MAGIC - `ingestion_date` : date to process (default = today). Pass specific date for backfill.
# MAGIC - `table_name`     : logical table name (default = all). Pass single name for targeted rerun.

#### COMMAND ----------
===============================================
##### CELL 1: Load shared config
===============================================
%run pulls all variables and functions from shared config into this notebook scope
Think of it like a Python import — but for Databricks notebooks

In [0]:
%run "../config/00_shared_config"

In [0]:
# COMMAND ----------
# =============================================================================
# CELL 2: Notebook widgets (parameters passed by Databricks Jobs or manual run)
# =============================================================================
from pyspark.sql import functions as F, Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, DateType, DecimalType, BooleanType, LongType
)
from delta.tables import DeltaTable
from datetime import datetime, timezone

# Widgets let you pass parameters when running manually or via Databricks Jobs
dbutils.widgets.text("ingestion_date", "", "Ingestion Date (blank=today)")
dbutils.widgets.text("table_name",     "", "Table Name (blank=all tables)")

# Resolve parameter values
INGESTION_DATE = dbutils.widgets.get("ingestion_date").strip() or get_ingestion_date()
TARGET_TABLE   = dbutils.widgets.get("table_name").strip() or None   # None = process all

log(f"Starting Bronze→Silver | ingestion_date={INGESTION_DATE} | target_table={TARGET_TABLE or 'ALL'}")


In [0]:
# COMMAND ----------
# =============================================================================
# CELL 3: Schema definitions (Silver target schema per table)
# Enforcing schema here = no surprises in downstream Gold queries
# All timestamps come from Bronze as STRING → cast to TIMESTAMP here
# =============================================================================

SILVER_SCHEMAS = {

    "orders": StructType([
        StructField("order_id",                    StringType(),    False),
        StructField("customer_id",                 StringType(),    True),
        StructField("order_status",                StringType(),    True),
        StructField("order_purchase_ts",           TimestampType(), True),
        StructField("order_approved_at",           TimestampType(), True),
        StructField("order_delivered_carrier_ts",  TimestampType(), True),
        StructField("order_delivered_customer_ts", TimestampType(), True),
        StructField("order_estimated_delivery_ts", TimestampType(), True),
        StructField("ingestion_date",              StringType(),    True),
        StructField("silver_loaded_at",            TimestampType(), True),
        StructField("dq_warn_flag",                BooleanType(),   True),
    ]),

    "order_items": StructType([
        StructField("order_id",           StringType(),       False),
        StructField("order_item_id",      IntegerType(),      False),
        StructField("product_id",         StringType(),       True),
        StructField("seller_id",          StringType(),       True),
        StructField("shipping_limit_ts",  TimestampType(),    True),
        StructField("price",              DecimalType(10,2),  True),
        StructField("freight_value",      DecimalType(10,2),  True),
        StructField("total_item_value",   DecimalType(10,2),  True),
        StructField("ingestion_date",     StringType(),       True),
        StructField("silver_loaded_at",   TimestampType(),    True),
    ]),

    "order_payments": StructType([
        StructField("order_id",              StringType(),      False),
        StructField("payment_sequential",    IntegerType(),     False),
        StructField("payment_type",          StringType(),      True),
        StructField("payment_installments",  IntegerType(),     True),
        StructField("payment_value",         DecimalType(10,2), True),
        StructField("order_purchase_ts",     TimestampType(),   True),
        StructField("ingestion_date",        StringType(),      True),
        StructField("silver_loaded_at",      TimestampType(),   True),
    ]),

    "order_reviews": StructType([
        StructField("review_id",              StringType(),    True),
        StructField("order_id",               StringType(),    True),
        StructField("review_score",           IntegerType(),   True),
        StructField("review_comment_title",   StringType(),    True),
        StructField("review_comment_message", StringType(),    True),
        StructField("review_creation_ts",     TimestampType(), True),
        StructField("review_answer_ts",       TimestampType(), True),
        StructField("has_comment",            BooleanType(),   True),
        StructField("ingestion_date",         StringType(),    True),
        StructField("silver_loaded_at",       TimestampType(), True),
    ]),

    "customers": StructType([
        StructField("customer_id",              StringType(), False),
        StructField("customer_unique_id",       StringType(), True),
        StructField("customer_zip_code_prefix", StringType(), True),
        StructField("customer_city",            StringType(), True),
        StructField("customer_state",           StringType(), True),
        StructField("created_at",               TimestampType(), True),
        StructField("ingestion_date",           StringType(), True),
        StructField("silver_loaded_at",         TimestampType(), True),
    ]),

    "products": StructType([
        StructField("product_id",                  StringType(),  False),
        StructField("product_category_name_pt",    StringType(),  True),
        StructField("product_name_length",         IntegerType(), True),
        StructField("product_description_length",  IntegerType(), True),
        StructField("product_photos_qty",          IntegerType(), True),
        StructField("product_weight_g",            IntegerType(), True),
        StructField("product_length_cm",           IntegerType(), True),
        StructField("product_height_cm",           IntegerType(), True),
        StructField("product_width_cm",            IntegerType(), True),
        StructField("created_at",                  TimestampType(), True),
        StructField("ingestion_date",              StringType(),  True),
        StructField("silver_loaded_at",            TimestampType(), True),
    ]),

    "sellers": StructType([
        StructField("seller_id",               StringType(), False),
        StructField("seller_zip_code_prefix",  StringType(), True),
        StructField("seller_city",             StringType(), True),
        StructField("seller_state",            StringType(), True),
        StructField("created_at",              TimestampType(), True),
        StructField("ingestion_date",          StringType(), True),
        StructField("silver_loaded_at",        TimestampType(), True),
    ]),

    "geolocation": StructType([
        StructField("geolocation_zip_code_prefix", StringType(), False),
        StructField("geolocation_lat",             DoubleType(),  True),
        StructField("geolocation_lng",             DoubleType(),  True),
        StructField("geolocation_city",            StringType(),  True),
        StructField("geolocation_state",           StringType(),  True),
        StructField("ingestion_date",              StringType(),  True),
        StructField("silver_loaded_at",            TimestampType(), True),
    ]),

    "category_translation": StructType([
        StructField("product_category_name_pt", StringType(), False),
        StructField("product_category_name_en", StringType(), True),
        StructField("ingestion_date",            StringType(), True),
        StructField("silver_loaded_at",          TimestampType(), True),
    ]),
}

log("Schema definitions loaded.")

In [0]:
# COMMAND ----------
# =============================================================================
# CELL 4: Schema transformation functions
# Each table gets its own function — keeps transformation logic readable
# Bronze column names → Silver column names + type casting
# =============================================================================

def transform_orders(df):
    return df.select(
        F.trim(F.col("order_id")).alias("order_id"),
        F.trim(F.col("customer_id")).alias("customer_id"),
        F.trim(F.lower(F.col("order_status"))).alias("order_status"),
        F.to_timestamp("order_purchase_timestamp").alias("order_purchase_ts"),
        F.to_timestamp("order_approved_at").alias("order_approved_at"),
        F.to_timestamp("order_delivered_carrier_date").alias("order_delivered_carrier_ts"),
        F.to_timestamp("order_delivered_customer_date").alias("order_delivered_customer_ts"),
        F.to_timestamp("order_estimated_delivery_date").alias("order_estimated_delivery_ts"),
        F.col("ingestion_date"),
        F.current_timestamp().alias("silver_loaded_at"),
        F.lit(False).alias("dq_warn_flag"),   # updated by DQ layer
    )

def transform_order_items(df):
    return df.select(
        F.trim(F.col("order_id")).alias("order_id"),
        F.col("order_item_id").cast(IntegerType()),
        F.trim(F.col("product_id")).alias("product_id"),
        F.trim(F.col("seller_id")).alias("seller_id"),
        F.to_timestamp("shipping_limit_date").alias("shipping_limit_ts"),
        F.col("price").cast(DecimalType(10,2)),
        F.col("freight_value").cast(DecimalType(10,2)),
        (F.col("price").cast(DecimalType(10,2)) +
         F.col("freight_value").cast(DecimalType(10,2))).alias("total_item_value"),
        F.col("ingestion_date"),
        F.current_timestamp().alias("silver_loaded_at"),
    )

def transform_order_payments(df):
    return df.select(
        F.trim(F.col("order_id")).alias("order_id"),
        F.col("payment_sequential").cast(IntegerType()),
        F.trim(F.lower(F.col("payment_type"))).alias("payment_type"),
        F.coalesce(F.col("payment_installments").cast(IntegerType()), F.lit(0)).alias("payment_installments"),
        F.col("payment_value").cast(DecimalType(10,2)),
        F.to_timestamp("order_purchase_timestamp").alias("order_purchase_ts"),
        F.col("ingestion_date"),
        F.current_timestamp().alias("silver_loaded_at"),
    )

def transform_order_reviews(df):
    return df.select(
        F.trim(F.col("review_id")).alias("review_id"),
        F.trim(F.col("order_id")).alias("order_id"),
        F.col("review_score").cast(IntegerType()),
        F.trim(F.col("review_comment_title")).alias("review_comment_title"),
        F.trim(F.col("review_comment_message")).alias("review_comment_message"),
        F.to_timestamp("review_creation_date").alias("review_creation_ts"),
        F.to_timestamp("review_answer_timestamp").alias("review_answer_ts"),
        F.col("review_comment_message").isNotNull().alias("has_comment"),
        F.col("ingestion_date"),
        F.current_timestamp().alias("silver_loaded_at"),
    )

def transform_customers(df):
    return df.select(
        F.trim(F.col("customer_id")).alias("customer_id"),
        F.trim(F.col("customer_unique_id")).alias("customer_unique_id"),
        F.lpad(F.trim(F.col("customer_zip_code_prefix")), 5, "0").alias("customer_zip_code_prefix"),
        F.trim(F.lower(F.col("customer_city"))).alias("customer_city"),
        F.trim(F.upper(F.col("customer_state"))).alias("customer_state"),
        F.to_timestamp("created_at").alias("created_at"),
        F.col("ingestion_date"),
        F.current_timestamp().alias("silver_loaded_at"),
    )

def transform_products(df):
    return df.select(
        F.trim(F.col("product_id")).alias("product_id"),
        F.trim(F.lower(F.col("product_category_name"))).alias("product_category_name_pt"),
        F.col("product_name_length").cast(IntegerType()),
        F.col("product_description_length").cast(IntegerType()),
        F.coalesce(F.col("product_photos_qty").cast(IntegerType()), F.lit(0)).alias("product_photos_qty"),
        F.col("product_weight_g").cast(IntegerType()),
        F.col("product_length_cm").cast(IntegerType()),
        F.col("product_height_cm").cast(IntegerType()),
        F.col("product_width_cm").cast(IntegerType()),
        F.to_timestamp("created_at").alias("created_at"),
        F.col("ingestion_date"),
        F.current_timestamp().alias("silver_loaded_at"),
    )

def transform_sellers(df):
    return df.select(
        F.trim(F.col("seller_id")).alias("seller_id"),
        F.lpad(F.trim(F.col("seller_zip_code_prefix")), 5, "0").alias("seller_zip_code_prefix"),
        F.trim(F.lower(F.col("seller_city"))).alias("seller_city"),
        F.trim(F.upper(F.col("seller_state"))).alias("seller_state"),
        F.to_timestamp("created_at").alias("created_at"),
        F.col("ingestion_date"),
        F.current_timestamp().alias("silver_loaded_at"),
    )

def transform_geolocation(df):
    return df.select(
        F.lpad(F.trim(F.col("geolocation_zip_code_prefix")), 5, "0").alias("geolocation_zip_code_prefix"),
        F.col("geolocation_lat").cast(DoubleType()),
        F.col("geolocation_lng").cast(DoubleType()),
        F.trim(F.lower(F.col("geolocation_city"))).alias("geolocation_city"),
        F.trim(F.upper(F.col("geolocation_state"))).alias("geolocation_state"),
        F.col("ingestion_date"),
        F.current_timestamp().alias("silver_loaded_at"),
    )

def transform_category_translation(df):
    return df.select(
        F.trim(F.lower(F.col("product_category_name"))).alias("product_category_name_pt"),
        F.trim(F.lower(
            F.regexp_replace(F.col("product_category_name_english"), "_", " ")
        )).alias("product_category_name_en"),
        F.col("ingestion_date"),
        F.current_timestamp().alias("silver_loaded_at"),
    )

TRANSFORM_FN = {
    "orders":               transform_orders,
    "order_items":          transform_order_items,
    "order_payments":       transform_order_payments,
    "order_reviews":        transform_order_reviews,
    "customers":            transform_customers,
    "products":             transform_products,
    "sellers":              transform_sellers,
    "geolocation":          transform_geolocation,
    "category_translation": transform_category_translation,
}

log("Transformation functions registered.")

In [0]:
# COMMAND ----------
# =============================================================================
# CELL 5: DQ Rules engine
# Returns (df_pass, df_fail) tuple
# Each rule adds a "dq_fail_reason" string — multiple failures concatenated
# =============================================================================

VALID_ORDER_STATUSES = {
    "delivered", "shipped", "canceled", "unavailable",
    "invoiced", "processing", "created", "approved"
}
VALID_PAYMENT_TYPES = {
    "credit_card", "boleto", "voucher", "debit_card", "not_defined"
}

def apply_dq_rules(table_name: str, df):
    """
    Applies DQ rules for given table.
    Returns: (df_pass, df_quarantine)
    df_quarantine has extra column: dq_fail_reason (string)
    """
    # Start: every row has empty fail reason
    df = df.withColumn("dq_fail_reason", F.lit(""))

    # ----------------------------------------------------------------
    # Table-specific rules — accumulate fail reasons (don't short-circuit)
    # A row can fail multiple rules — we capture all
    # ----------------------------------------------------------------
    if table_name == "orders":
        # DQ-01: order_id NOT NULL
        df = df.withColumn("dq_fail_reason", F.when(
            F.col("order_id").isNull(),
            F.concat(F.col("dq_fail_reason"), F.lit("DQ-01:order_id is NULL; "))
        ).otherwise(F.col("dq_fail_reason")))

        # DQ-02: order_status valid values
        df = df.withColumn("dq_fail_reason", F.when(
            ~F.col("order_status").isin(list(VALID_ORDER_STATUSES)),
            F.concat(F.col("dq_fail_reason"), F.lit("DQ-02:invalid order_status; "))
        ).otherwise(F.col("dq_fail_reason")))

        # DQ-03: order_purchase_ts not in future
        df = df.withColumn("dq_fail_reason", F.when(
            F.col("order_purchase_ts") > F.current_timestamp(),
            F.concat(F.col("dq_fail_reason"), F.lit("DQ-03:future order_purchase_ts; "))
        ).otherwise(F.col("dq_fail_reason")))

        # DQ-04: customer_id not null (warn only — doesn't quarantine)
        df = df.withColumn("dq_warn_flag", F.when(
            F.col("customer_id").isNull(), F.lit(True)
        ).otherwise(F.col("dq_warn_flag")))

    elif table_name == "order_items":
        # DQ-05: order_id, product_id, seller_id NOT NULL
        df = df.withColumn("dq_fail_reason", F.when(
            F.col("order_id").isNull() |
            F.col("product_id").isNull() |
            F.col("seller_id").isNull(),
            F.concat(F.col("dq_fail_reason"), F.lit("DQ-05:key columns NULL; "))
        ).otherwise(F.col("dq_fail_reason")))

        # DQ-06: price > 0, freight >= 0
        df = df.withColumn("dq_fail_reason", F.when(
            (F.col("price") <= 0) | (F.col("freight_value") < 0),
            F.concat(F.col("dq_fail_reason"), F.lit("DQ-06:invalid price/freight; "))
        ).otherwise(F.col("dq_fail_reason")))

    elif table_name == "order_payments":
        # DQ-07: payment_value >= 0
        df = df.withColumn("dq_fail_reason", F.when(
            F.col("payment_value") < 0,
            F.concat(F.col("dq_fail_reason"), F.lit("DQ-07:negative payment_value; "))
        ).otherwise(F.col("dq_fail_reason")))

        # DQ-08: payment_type valid (warn only)
        df = df.withColumn("dq_fail_reason", F.when(
            ~F.col("payment_type").isin(list(VALID_PAYMENT_TYPES)),
            F.concat(F.col("dq_fail_reason"), F.lit("DQ-08:invalid payment_type(warn); "))
        ).otherwise(F.col("dq_fail_reason")))

    elif table_name == "order_reviews":
        # DQ-11: review_score BETWEEN 1 AND 5
        df = df.withColumn("dq_fail_reason", F.when(
            F.col("review_score").isNull() |
            (F.col("review_score") < 1) |
            (F.col("review_score") > 5),
            F.concat(F.col("dq_fail_reason"), F.lit("DQ-11:invalid review_score; "))
        ).otherwise(F.col("dq_fail_reason")))

        # review_id NULL quarantine (PROB-03)
        df = df.withColumn("dq_fail_reason", F.when(
            F.col("review_id").isNull(),
            F.concat(F.col("dq_fail_reason"), F.lit("DQ-REV-01:review_id is NULL; "))
        ).otherwise(F.col("dq_fail_reason")))

    elif table_name == "customers":
        # DQ-09: customer_id NOT NULL
        df = df.withColumn("dq_fail_reason", F.when(
            F.col("customer_id").isNull(),
            F.concat(F.col("dq_fail_reason"), F.lit("DQ-09:customer_id is NULL; "))
        ).otherwise(F.col("dq_fail_reason")))

    elif table_name == "products":
        # DQ-12: product_id NOT NULL
        df = df.withColumn("dq_fail_reason", F.when(
            F.col("product_id").isNull(),
            F.concat(F.col("dq_fail_reason"), F.lit("DQ-12:product_id is NULL; "))
        ).otherwise(F.col("dq_fail_reason")))

    elif table_name == "geolocation":
        # DQ-13: lat BETWEEN -90 AND 90 (PROB-05 detection)
        df = df.withColumn("dq_fail_reason", F.when(
            (F.col("geolocation_lat") < -90) | (F.col("geolocation_lat") > 90),
            F.concat(F.col("dq_fail_reason"), F.lit("DQ-13:invalid geolocation_lat; "))
        ).otherwise(F.col("dq_fail_reason")))

        # DQ-14: lng BETWEEN -180 AND 180
        df = df.withColumn("dq_fail_reason", F.when(
            (F.col("geolocation_lng") < -180) | (F.col("geolocation_lng") > 180),
            F.concat(F.col("dq_fail_reason"), F.lit("DQ-14:invalid geolocation_lng; "))
        ).otherwise(F.col("dq_fail_reason")))

    # ----------------------------------------------------------------
    # Split: pass = empty fail reason, fail = non-empty
    # ----------------------------------------------------------------
    df_pass = df.filter(F.col("dq_fail_reason") == "").drop("dq_fail_reason")
    df_fail = df.filter(F.col("dq_fail_reason") != "")

    return df_pass, df_fail

log("DQ rules engine loaded.")

In [0]:
# COMMAND ----------
# =============================================================================
# CELL 6: Deduplication function
# Applied AFTER schema transform, BEFORE DQ rules
# Keeps latest row per PK based on available timestamp column
# =============================================================================

def deduplicate(df, pk_cols: list, order_col: str = None):
    """
    Deduplicates DataFrame keeping latest row per pk_cols.
    If order_col is None, uses silver_loaded_at (current_timestamp) as tiebreaker.
    Important: for geolocation, uses median lat/lng aggregation instead.
    """
    if not order_col:
        order_col = "silver_loaded_at"

    w = Window.partitionBy(*pk_cols).orderBy(F.desc(order_col))
    return (
        df.withColumn("_rn", F.row_number().over(w))
          .filter(F.col("_rn") == 1)
          .drop("_rn")
    )

def deduplicate_geolocation(df):
    """
    Special dedup for geolocation — 1M rows, 99%+ duplicates per zip.
    Uses median lat/lng to avoid outlier coordinates skewing the location.
    """
    return df.groupBy("geolocation_zip_code_prefix").agg(
        F.percentile_approx("geolocation_lat", 0.5).alias("geolocation_lat"),
        F.percentile_approx("geolocation_lng", 0.5).alias("geolocation_lng"),
        F.first("geolocation_city").alias("geolocation_city"),
        F.first("geolocation_state").alias("geolocation_state"),
        F.first("ingestion_date").alias("ingestion_date"),
        F.first("silver_loaded_at").alias("silver_loaded_at"),
    )

log("Deduplication functions loaded.")

In [0]:
# COMMAND ----------
# =============================================================================
# CELL 7: Delta MERGE function (upsert)
# Why MERGE not INSERT:
#   - Idempotent: same data in = same data out, no duplicates
#   - Handles late-arriving data
#   - If ADF re-runs same partition (bug/retry), Silver stays clean
# =============================================================================

def merge_into_silver(table_name: str, df_pass, pk_cols: list):
    """
    MERGE df_pass into Silver Delta table.
    If Silver table doesn't exist → CREATE it (first run).
    If it exists → MERGE (upsert on PK).
    """
    silver_table_path = f"{SILVER_BASE}/{table_name}"
    silver_table_uc   = f"{UC_SILVER}.{table_name}"

    # Check if Silver table already exists
    table_exists = spark.catalog.tableExists(silver_table_uc)

    if not table_exists:
        log(f"  Silver table {silver_table_uc} does not exist. Creating...")
        (df_pass.write
             .format("delta")
             .mode("overwrite")
             .option("path", silver_table_path)
             .saveAsTable(silver_table_uc))
        log(f"  Created {silver_table_uc} with {df_pass.count()} rows.")
        return df_pass.count()

    # Table exists → MERGE
    delta_table = DeltaTable.forName(spark, silver_table_uc)

    # Build merge condition on composite PK
    merge_condition = " AND ".join([
        f"target.{col} = source.{col}" for col in pk_cols
    ])

    # Get all columns for update map (exclude PK cols from update to be safe)
    update_cols = [c for c in df_pass.columns if c not in pk_cols]
    update_map  = {col: f"source.{col}" for col in update_cols}

    (delta_table.alias("target")
        .merge(df_pass.alias("source"), merge_condition)
        .whenMatchedUpdate(set=update_map)
        .whenNotMatchedInsertAll()
        .execute())

    rows_affected = df_pass.count()
    log(f"  MERGE complete for {silver_table_uc}: {rows_affected} rows upserted.")
    return rows_affected

log("MERGE function loaded.")

In [0]:
# COMMAND ----------
# =============================================================================
# CELL 8: Quarantine write function
# Appends failed records to quarantine Delta table
# Each table has its own quarantine table
# =============================================================================

def write_to_quarantine(table_name: str, df_fail, ingestion_date: str):
    """
    Appends DQ-failed records to quarantine Delta table.
    Adds metadata columns: source_table, ingestion_date, quarantine_loaded_at.
    """
    if df_fail.count() == 0:
        log(f"  No quarantine records for {table_name}. Skipping.")
        return 0

    quarantine_table_uc   = f"{UC_QUARANTINE}.{table_name}"
    quarantine_table_path = f"{QUARANTINE_BASE}/{table_name}"

    df_quarantine = (df_fail
        .withColumn("source_table",          F.lit(table_name))
        .withColumn("ingestion_date",        F.lit(ingestion_date))
        .withColumn("quarantine_loaded_at",  F.current_timestamp())
    )

    table_exists = spark.catalog.tableExists(quarantine_table_uc)

    write_mode = "append" if table_exists else "overwrite"

    (df_quarantine.write
         .format("delta")
         .mode(write_mode)
         .option("mergeSchema", "true")
         .option("path", quarantine_table_path)
         .saveAsTable(quarantine_table_uc))

    count = df_fail.count()
    log(f"  Quarantine: {count} records written to {quarantine_table_uc}.", level="WARN")
    return count

log("Quarantine function loaded.")

In [0]:
# COMMAND ----------
# =============================================================================
# CELL 9: Main processing loop
# Processes each table end-to-end
# =============================================================================

# Determine which tables to process
tables_to_process = (
    [TARGET_TABLE] if TARGET_TABLE
    else list(TABLE_REGISTRY.keys())
)

log(f"Processing {len(tables_to_process)} tables: {tables_to_process}")

# Run summary — populated as tables process
run_summary = []

for tbl in tables_to_process:
    log(f"\n{'='*60}")
    log(f"Processing: {tbl}")
    log(f"{'='*60}")

    cfg = TABLE_REGISTRY[tbl]

    # ------------------------------------------------------------------
    # STEP A: Read Bronze partition
    # ------------------------------------------------------------------
    bronze_path = f"{BRONZE_BASE}/{cfg['bronze_folder']}/ingestion_date={INGESTION_DATE}*"
    bronze_path
    log(f"  Reading Bronze: {bronze_path}")

    try:
        df_raw = spark.read.parquet(bronze_path)
    except Exception as e:
        log(f"  WARN: No Bronze data found for {tbl} on {INGESTION_DATE}. Skipping. [{e}]", level="WARN")
        run_summary.append({
            "table": tbl, "bronze_rows": 0, "pass_rows": 0,
            "quarantine_rows": 0, "status": "SKIPPED_NO_DATA"
        })
        continue

    bronze_count = df_raw.count()
    log(f"  Bronze rows read: {bronze_count:,}")

    if bronze_count == 0:
        log(f"  Empty Bronze partition. Skipping.", level="WARN")
        run_summary.append({
            "table": tbl, "bronze_rows": 0, "pass_rows": 0,
            "quarantine_rows": 0, "status": "SKIPPED_EMPTY"
        })
        continue

    # ------------------------------------------------------------------
    # STEP B: Transform (Bronze column names → Silver column names + types)
    # ------------------------------------------------------------------
    log(f"  Applying schema transformation...")
    transform_fn = TRANSFORM_FN.get(tbl)
    if not transform_fn:
        log(f"  ERROR: No transform function for {tbl}", level="ERROR")
        continue

    df_transformed = transform_fn(df_raw)

    # ------------------------------------------------------------------
    # STEP C: Deduplication
    # ------------------------------------------------------------------
    log(f"  Deduplicating on PK: {cfg['pk_cols']}...")
    if tbl == "geolocation":
        df_deduped = deduplicate_geolocation(df_transformed)
    else:
        df_deduped = deduplicate(
            df_transformed,
            pk_cols=cfg["pk_cols"],
            order_col=cfg.get("watermark_col")
        )
    dedup_count = df_deduped.count()
    log(f"  After dedup: {dedup_count:,} rows (removed {bronze_count - dedup_count:,} duplicates)")

    # ------------------------------------------------------------------
    # STEP D: Apply DQ Rules
    # ------------------------------------------------------------------
    log(f"  Applying DQ rules...")
    df_pass, df_fail = apply_dq_rules(tbl, df_deduped)
    pass_count  = df_pass.count()
    fail_count  = df_fail.count()
    log(f"  DQ result: {pass_count:,} pass | {fail_count:,} quarantine")

    if fail_count > 0:
        # Show sample of failures for visibility
        log(f"  Sample quarantine reasons:")
        (df_fail
            .groupBy("dq_fail_reason")
            .count()
            .orderBy(F.desc("count"))
            .show(10, truncate=False))

    # ------------------------------------------------------------------
    # STEP E: MERGE into Silver
    # ------------------------------------------------------------------
    log(f"  Writing to Silver...")
    silver_rows = merge_into_silver(tbl, df_pass, cfg["pk_cols"])

    # ------------------------------------------------------------------
    # STEP F: Write quarantine
    # ------------------------------------------------------------------
    quarantine_rows = write_to_quarantine(tbl, df_fail, INGESTION_DATE)

    run_summary.append({
        "table":          tbl,
        "bronze_rows":    bronze_count,
        "after_dedup":    dedup_count,
        "pass_rows":      pass_count,
        "quarantine_rows": quarantine_rows,
        "status":         "SUCCESS"
    })

log("\nAll tables processed.")

In [0]:
# COMMAND ----------
# =============================================================================
# CELL 10: Run summary
# =============================================================================
import pandas as pd

summary_df = pd.DataFrame(run_summary)
print("\n" + "="*70)
print("BRONZE → SILVER RUN SUMMARY")
print("="*70)
print(summary_df.to_string(index=False))
print("="*70)

# Fail if any quarantine rate > 10% of bronze rows (configurable threshold)
QUARANTINE_THRESHOLD = 0.10
alerts = []
for row in run_summary:
    if row["bronze_rows"] > 0:
        q_rate = row["quarantine_rows"] / row["bronze_rows"]
        if q_rate > QUARANTINE_THRESHOLD:
            alerts.append(
                f"ALERT: {row['table']} quarantine rate = {q_rate:.1%} "
                f"({row['quarantine_rows']:,} / {row['bronze_rows']:,})"
            )

if alerts:
    log("\n".join(alerts), level="WARN")
    # In Phase 5 we'll wire this to email/Teams alert
    # For now: raise exception so Databricks Job marks run as FAILED
    # Comment out the raise if you want pipeline to continue despite high quarantine
    # raise Exception(f"Quarantine threshold exceeded: {alerts}")

log("Bronze → Silver complete.")
